# PC-GNN em um terceiro conjunto de dados: S-FFSD (v4)

Disciplina de Detecção de Intrusão. Continuação de `pcgnn_replicacao_artigo_v3.ipynb`, que reproduziu o PC-GNN
(Liu et al., WWW 2021) no YelpChi e na Amazon. Este notebook leva o mesmo método a um conjunto **que o artigo não usou**,
o **S-FFSD** (transações de cartão de crédito, simulado, do repositório `AI4Risk/antifraud`), e responde a duas perguntas:

1. **O PC-GNN, com a configuração dos autores, funciona nesse conjunto?** (linha de base, 10 sementes)
2. **Ajustar os hiperparâmetros e o limiar melhora, e a melhora se sustenta em sementes novas?** (busca na validação, depois 10 sementes no teste)

**O que o notebook faz**
1. Clona o código dos autores no commit `9d7d7fa` e cria uma **cópia adaptada** (o clone original não é tocado). A adaptação generaliza o agregador de 3 para R relações (o S-FFSD tem 4), aceita o nome `sffsd`, expõe `keep_ratio` e `feat_norm` e grava as probabilidades de validação e teste. Com 3 relações, o agregador adaptado devolve **exatamente** as mesmas saídas do original (testado fora do Colab, diferença 0,0).
2. Baixa o S-FFSD, confere o SHA-256 e mostra as estatísticas do conjunto e do grafo.
3. Gera os atributos de janela de tempo do `antifraud` com uma versão vetorizada (a original leva horas), verificada contra a original.
4. Treina o PC-GNN padrão (YAML do YelpChi, sem alterações) com 10 sementes.
5. Busca de hiperparâmetros **só com a validação** (AUC de validação), confirma os melhores candidatos em mais sementes e roda a configuração escolhida com as mesmas 10 sementes da linha de base.
6. Escolhe o limiar de decisão na validação e mostra o efeito do limiar separado do efeito dos hiperparâmetros.
7. Compara com modelos que não usam o grafo (gradient boosting e regressão logística nos mesmos atributos).
8. Exporta tabelas, figura e um **resumo em texto para você colar na conversa**.

**Decisões que valem para o relatório**
- Nós **sem rótulo** (rótulo 2, 62% do conjunto) ficam no grafo e participam da agregação, mas ficam fora da divisão e das métricas, como o código dos autores faz com a Amazon.
- A divisão é a do código dos autores: estratificada, 40% treino, restante 33% validação e 67% teste, `random_state=2`, sobre os nós rotulados. É uma divisão **aleatória**, e as transações são temporais; isso pode ser otimista e é uma limitação a registrar.
- Atributos: `Amount` e os 120 atributos de janela de tempo. **`Time` e os códigos de `Source/Target/Location/Type` ficam de fora por padrão** (inteiros ordinais sem significado); o `antifraud` os mantém. Mude `INCLUDE_ID_FEATURES` para testar.
- Atributos em log com sinal e min-max em [0, 1], para casar com a normalização por linha do código original.
- Grafo igual ao do `antifraud`: quatro relações (mesmo Source, Target, Location, Type), cada transação ligada às **2 seguintes** do mesmo grupo, simetrizado, com laço próprio.

Execute de cima para baixo. Cada execução grava `result.json` no Drive; uma sessão interrompida retoma de onde parou.

## 1. Controles

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from concurrent.futures import ThreadPoolExecutor, as_completed
import hashlib
import json
import math
import os
import platform
import random
import re
import shutil
import subprocess
import sys
import time
import warnings
from urllib.request import urlretrieve
from zipfile import ZipFile

# ---------- O que executar ----------
SEEDS = [42, 448, 854, 29493, 88867, 72, 123, 2024, 31337, 7]   # sementes finais (linha de base e configuração ajustada)
TUNE_SEEDS = [1001]                      # semente(s) da busca; distintas das finais para a escolha não ver as sementes do resultado
CONFIRM_EXTRA_SEEDS = [1002, 1003]       # sementes extras para confirmar os melhores candidatos da busca
N_RANDOM_CONFIGS = 12                    # nº de configurações sorteadas na busca
TOP_K = 3                                # quantas passam para a confirmação
TUNE_TIME_BUDGET_MIN = None              # None = sem limite. Ex.: 150 para parar a busca depois de ~2,5 h
N_PARALLEL = min(2, os.cpu_count() or 1) # execuções simultâneas (o gargalo é o laço Python; o Colab gratuito tem 2 CPUs)
INCLUDE_ID_FEATURES = False              # True = inclui Time e os códigos de Source/Target/Location/Type (como o antifraud)
EPOCHS_OVERRIDE = None                   # None = 51 épocas do YAML (a busca pode escolher 31 ou 51)
QUICK_TEST = False                       # True = teste rápido do notebook (2 sementes, 2 épocas, 2 configurações)

# ---------- Código e dados ----------
WORK = Path(os.environ.get('WORK_ROOT', '/content'))
REPO_URL = 'https://github.com/Fialhoalves/PC-GNN.git'
REPO_COMMIT = '9d7d7fae491081178b6e12e193fb89cfc330f2be'     # o mesmo commit da replicação (v3)
REPO = WORK / 'PC-GNN'
PATCHED = WORK / 'PC-GNN-sffsd'
SUPPORT = WORK / 'sffsd_support'
SFFSD_URL = 'https://raw.githubusercontent.com/AI4Risk/antifraud/main/data/S-FFSD.zip'
SFFSD_CSV_SHA256 = 'a2d78b983dfacaae394e4d69ece46ae880fd6bd4761bc5ee759977ba12b185c6'
USE_DRIVE = True
DATA_DRIVE_DIR = Path('/content/drive/MyDrive/PC_GNN_data')          # opcional: S-FFSD.csv ou S-FFSD.zip aqui tem prioridade
RESULTS_ROOT = Path('/content/drive/MyDrive/PC_GNN_sffsd')           # resultados, logs e relatórios
DATA_DIR = WORK / 'sffsd_data'                                       # arrays e listas de adjacência gerados

warnings.filterwarnings('ignore', category=DeprecationWarning)
os.environ['PYTHONWARNINGS'] = 'ignore::DeprecationWarning'

if QUICK_TEST:
    SEEDS, TUNE_SEEDS, CONFIRM_EXTRA_SEEDS = SEEDS[:2], TUNE_SEEDS[:1], CONFIRM_EXTRA_SEEDS[:1]
    N_RANDOM_CONFIGS, TOP_K, EPOCHS_OVERRIDE = 2, 2, 2
assert len(set(SEEDS)) == len(SEEDS) >= 2 and not set(SEEDS) & set(TUNE_SEEDS + CONFIRM_EXTRA_SEEDS), \
    'SEEDS: pelo menos 2 distintas, e disjuntas das sementes da busca.'

## 2. Ambiente e Drive
Usa o PyTorch que já vem no Colab (não instale o `requirements.txt` do repositório). O treino roda em CPU: o gargalo é o laço
Python de amostragem de vizinhos. Com `N_PARALLEL = 2`, duas execuções rodam ao mesmo tempo, cada uma com 1 *thread* de BLAS.

In [ ]:
import importlib
import importlib.metadata

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if USE_DRIVE and IN_COLAB:
    drive.mount('/content/drive')
    if not Path('/content/drive/MyDrive').is_dir():
        raise RuntimeError('Drive indisponível; confira a conta Google e a montagem.')
else:
    USE_DRIVE = False
    DATA_DRIVE_DIR = None
    RESULTS_ROOT = WORK / 'PC_GNN_sffsd'

if QUICK_TEST:
    RESULTS_ROOT = RESULTS_ROOT / 'quick_test'
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

PACKAGES = {'numpy': 'numpy', 'scipy': 'scipy', 'sklearn': 'scikit-learn', 'torch': 'torch',
            'yaml': 'PyYAML', 'pandas': 'pandas', 'matplotlib': 'matplotlib'}
VERSIONS = {}
for module, package in PACKAGES.items():
    importlib.import_module(module)
    VERSIONS[package] = importlib.metadata.version(package)

import numpy as np
import pandas as pd
import yaml
import matplotlib.pyplot as plt
import scipy.sparse as sp
import torch
from scipy.stats import wilcoxon, spearmanr
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, confusion_matrix
try:
    from IPython.display import display
except ImportError:
    display = print

print('Python', sys.version.split()[0], '|', platform.platform(), '| CPUs:', os.cpu_count())
print(json.dumps(VERSIONS, indent=2))
print('Resultados em:', RESULTS_ROOT)
print('Execuções em paralelo:', N_PARALLEL)

Mounted at /content/drive
Python 3.13.18 | Linux-6.6.159+-x86_64-with-glibc2.39 | CPUs: 2
{
  "numpy": "2.4.6",
  "scipy": "1.16.3",
  "scikit-learn": "1.6.1",
  "torch": "2.14.1+cpu",
  "PyYAML": "6.0.3",
  "pandas": "2.2.3",
  "matplotlib": "3.10.0"
}
Resultados em: /content/drive/MyDrive/PC_GNN_sffsd
Execuções em paralelo: 2


## 3. Código dos autores e cópia adaptada
O repositório é clonado e posto **no mesmo commit da replicação** (`9d7d7fa`). O clone não é alterado: a adaptação é aplicada a uma cópia (`PC-GNN-sffsd`) por um script que **falha** se algum trecho esperado do código não for encontrado exatamente uma vez. O `diff` entre o original e a cópia é salvo em `RESULTS_ROOT/patch.diff` para auditoria.

Mudanças (todas em `src/`): `layers.py` (agregador inter-relação com R relações em vez de 3 fixas; fração de vizinhos mantida `keep_ratio`, que no original é 0,5 fixo), `model_handler.py` (nome `sffsd` e divisão só com nós rotulados; `feat_norm` para ligar ou desligar a normalização por linha; um agregador por relação; gravação das probabilidades), `utils.py` (leitura do S-FFSD e `dump_probs`).

In [ ]:
# o %%writefile das duas células seguintes não cria a pasta
Path('/content/sffsd_support').mkdir(parents=True, exist_ok=True)

In [ ]:
%%writefile /content/sffsd_support/sffsd_patch.py
import re
import shutil
from pathlib import Path


def _sub_once(text, pattern, replacement, label, flags=0):
    new, count = re.subn(pattern, lambda m: replacement, text, count=0, flags=flags)
    if count != 1:
        raise RuntimeError(f'Patch "{label}": esperava 1 ocorrência, encontrou {count}. O código do repositório mudou?')
    return new


NEW_INTER_FORWARD = '''	def forward(self, nodes, labels, train_flag=True):
		"""
		Mesma lógica do original, generalizada para R relações (o original fixa R = 3).
		"""
		# extract 1-hop neighbor ids from adj lists of each single-relation graph
		to_neighs = []
		for adj_list in self.adj_lists:
			to_neighs.append([set(adj_list[int(node)]) for node in nodes])

		# find unique nodes and their neighbors used in current batch
		unique_nodes = set.union(*[set.union(*neighs) for neighs in to_neighs], set(nodes))

		# calculate label-aware scores
		if self.cuda:
			batch_features = self.features(torch.cuda.LongTensor(list(unique_nodes)))
			pos_features = self.features(torch.cuda.LongTensor(list(self.train_pos)))
		else:
			batch_features = self.features(torch.LongTensor(list(unique_nodes)))
			pos_features = self.features(torch.LongTensor(list(self.train_pos)))
		batch_scores = self.label_clf(batch_features)
		pos_scores = self.label_clf(pos_features)
		id_mapping = {node_id: index for node_id, index in zip(unique_nodes, range(len(unique_nodes)))}

		# the label-aware scores for current batch of nodes
		center_scores = batch_scores[itemgetter(*nodes)(id_mapping), :]

		rel_feats = []
		for r, intra_agg in enumerate(self.intra_aggs):
			# get neighbor node id list for each batch node in relation r
			r_list = [list(to_neigh) for to_neigh in to_neighs[r]]
			# assign label-aware scores to neighbor nodes
			r_scores = [batch_scores[itemgetter(*to_neigh)(id_mapping), :].view(-1, 2) for to_neigh in r_list]
			# number of neighbors kept for aggregation
			r_sample_num_list = [math.ceil(len(neighs) * self.thresholds[r]) for neighs in r_list]
			# intra-aggregation step, Eq. (8) in the paper
			r_feats, _ = intra_agg.forward(nodes, labels, r_list, center_scores, r_scores, pos_scores, r_sample_num_list, train_flag)
			rel_feats.append(r_feats)

		# get features or embeddings for batch nodes
		if self.cuda and isinstance(nodes, list):
			index = torch.LongTensor(nodes).cuda()
		else:
			index = torch.LongTensor(nodes)
		self_feats = self.features(index)

		# concat the intra-aggregated embeddings from each relation, Eq. (9) in the paper
		cat_feats = torch.cat([self_feats] + rel_feats, dim=1)

		combined = F.relu(cat_feats.mm(self.weight).t())

		return combined, center_scores


'''

DUMP_PROBS = '''

def dump_probs(path, model, sets, batch_size):
	"""
	Salva rótulos e probabilidades (sigmoide do logit da classe 1, como no teste do código original)
	de cada conjunto em um .npz, para permitir escolher o limiar fora do treino.
	sets: {'valid': (idx, y), 'test': (idx, y)}
	"""
	out = {}
	for name, (idx, y) in sets.items():
		probs = []
		for start in range(0, len(idx), batch_size):
			batch_nodes = idx[start:start + batch_size]
			batch_label = y[start:start + batch_size]
			gnn_prob, _ = model.to_prob(batch_nodes, batch_label, train_flag=False)
			probs.extend(gnn_prob.data.cpu().numpy()[:, 1].tolist())
		out[f'{name}_y'] = np.asarray(y)
		out[f'{name}_p'] = np.asarray(probs)
	np.savez(path, **out)
'''

SFFSD_LOAD = '''	elif data == 'sffsd':
		# S-FFSD: arrays e listas de adjacência gerados pelo notebook (ver sffsd_manifest.json)
		import json
		manifest = json.loads(open(prefix + 'sffsd_manifest.json').read())
		labels = np.load(prefix + 'sffsd_labels.npy').flatten()
		feat_data = np.load(prefix + 'sffsd_features.npy')
		graphs = []
		for name in ['homo'] + manifest['relations']:
			with open(prefix + f'sffsd_{name}_adjlists.pickle', 'rb') as file:
				graphs.append(pickle.load(file))
		return graphs, feat_data, labels

'''

SFFSD_SPLIT = '''		elif args.data_name == 'sffsd':  # S-FFSD: rótulo 2 = sem rótulo, fora da divisão
			labeled = np.where(labels < 2)[0]
			index = labeled.tolist()
			idx_train, idx_rest, y_train, y_rest = train_test_split(index, labels[labeled], stratify=labels[labeled],
																	train_size=args.train_ratio, random_state=2, shuffle=True)
			idx_valid, idx_test, y_valid, y_test = train_test_split(idx_rest, y_rest, stratify=y_rest,
																	test_size=args.test_ratio, random_state=2, shuffle=True)

'''


def apply_patch(src_repo, dst_repo):
    """Copia o repositório (sem .git) e aplica as adaptações para o S-FFSD. O original não é modificado."""
    src_repo, dst_repo = Path(src_repo), Path(dst_repo)
    if dst_repo.exists():
        shutil.rmtree(dst_repo)
    shutil.copytree(src_repo, dst_repo, ignore=shutil.ignore_patterns('.git', '__pycache__', 'data', 'pytorch_models'))

    # ---- src/utils.py ----
    path = dst_repo / 'src' / 'utils.py'
    text = path.read_text(encoding='utf-8')
    text = _sub_once(text, r"\telif data == 'amazon':", SFFSD_LOAD + "\telif data == 'amazon':", 'utils.load_data')
    text += DUMP_PROBS
    path.write_text(text, encoding='utf-8')

    # ---- src/layers.py ----
    path = dst_repo / 'src' / 'layers.py'
    text = path.read_text(encoding='utf-8')
    head, rest = text.split('class IntraAgg', 1)
    fwd_start = head.index('\tdef forward(self, nodes, labels, train_flag=True):')
    init = head[:fwd_start]
    init = _sub_once(init, r"train_pos, adj_lists, intraggs, inter='GNN', cuda=True\):",
                     "train_pos, adj_lists, intraggs, inter='GNN', cuda=True, keep_ratio=0.5):", 'layers.init-signature')
    init = _sub_once(init, r"\t\tself\.intra_agg1 = intraggs\[0\]\n\t\tself\.intra_agg2 = intraggs\[1\]\n\t\tself\.intra_agg3 = intraggs\[2\]\n",
                     "\t\tself.intra_aggs = nn.ModuleList(intraggs)\n", 'layers.intra-assign')
    init = _sub_once(init, r"\t\tself\.intra_agg1\.cuda = cuda\n\t\tself\.intra_agg2\.cuda = cuda\n\t\tself\.intra_agg3\.cuda = cuda\n",
                     "\t\tfor intra_agg in self.intra_aggs:\n\t\t\tintra_agg.cuda = cuda\n", 'layers.intra-cuda')
    init = _sub_once(init, r"\t\tself\.thresholds = \[0\.5, 0\.5, 0\.5\]\n",
                     "\t\tself.thresholds = [keep_ratio] * len(intraggs)\n", 'layers.thresholds')
    path.write_text(init + NEW_INTER_FORWARD + 'class IntraAgg' + rest, encoding='utf-8')

    # ---- src/model_handler.py ----
    path = dst_repo / 'src' / 'model_handler.py'
    text = path.read_text(encoding='utf-8')
    text = _sub_once(text, r"\t\t\[homo, relation1, relation2, relation3\], feat_data, labels = load_data\(args\.data_name, prefix=args\.data_dir\)\n",
                     "\t\tgraphs, feat_data, labels = load_data(args.data_name, prefix=args.data_dir)\n\t\thomo, relations = graphs[0], graphs[1:]\n",
                     'handler.load')
    text = _sub_once(text, r"\t\tprint\(f'Run on \{args\.data_name\}", SFFSD_SPLIT + "\t\tprint(f'Run on {args.data_name}", 'handler.split')
    text = _sub_once(text, r"\t\tfeat_data = normalize\(feat_data\)\n",
                     "\t\tif getattr(args, 'feat_norm', 'row') == 'row':\n\t\t\tfeat_data = normalize(feat_data)\n", 'handler.normalize')
    text = _sub_once(text, r"\t\t\tadj_lists = \[relation1, relation2, relation3\]\n", "\t\t\tadj_lists = list(relations)\n", 'handler.adj')
    text = _sub_once(
        text,
        r"\t\t\tintra1 = IntraAgg\(.*?\n\t\t\tintra2 = IntraAgg\(.*?\n\t\t\tintra3 = IntraAgg\(.*?\n"
        r"\t\t\tinter1 = InterAgg\(.*?\n.*?adj_lists, \[intra1, intra2, intra3\], inter=args\.multi_relation, cuda=args\.cuda\)\n",
        "\t\t\tintras = [IntraAgg(features, feat_data.shape[1], args.emb_size, self.dataset['train_pos'], args.rho, cuda=args.cuda)\n"
        "\t\t\t\t\t  for _ in adj_lists]\n"
        "\t\t\tinter1 = InterAgg(features, feat_data.shape[1], args.emb_size, self.dataset['train_pos'],\n"
        "\t\t\t\t\t\t\t  adj_lists, intras, inter=args.multi_relation, cuda=args.cuda,\n"
        "\t\t\t\t\t\t\t  keep_ratio=getattr(args, 'keep_ratio', 0.5))\n",
        'handler.aggregators', flags=re.S)
    text = _sub_once(
        text,
        r"\t\treturn f1_mac_test, f1_1_test, f1_0_test, auc_test, gmean_test\n",
        "\t\tif getattr(args, 'dump_probs', None) and args.model == 'PCGNN':\n"
        "\t\t\tdump_probs(args.dump_probs, gnn_model, {'valid': (idx_valid, y_valid), 'test': (idx_test, y_test)}, args.batch_size)\n"
        "\t\treturn f1_mac_test, f1_1_test, f1_0_test, auc_test, gmean_test\n",
        'handler.dump')
    text = _sub_once(text, r"from src\.utils import test_pcgnn, test_sage, load_data, pos_neg_split, normalize, pick_step",
                     "from src.utils import test_pcgnn, test_sage, load_data, pos_neg_split, normalize, pick_step, dump_probs", 'handler.import')
    path.write_text(text, encoding='utf-8')
    return dst_repo

Writing /content/sffsd_support/sffsd_patch.py


In [ ]:
%%writefile /content/sffsd_support/sffsd_data.py
from collections import Counter, deque
import numpy as np
import pandas as pd
import scipy.sparse as sp

TIME_SPAN = [2, 3, 5, 15, 20, 50, 100, 150, 200, 300, 864, 2590, 5100, 10000, 24000]   # as do antifraud (feature_engineering/data_process.py)
RELATIONS = ['Source', 'Target', 'Location', 'Type']
EDGES_PER_TRANS = 3      # no antifraud o laço é range(3): laço próprio + as 2 transações seguintes do mesmo grupo


def window_distinct(codes, span):
    """Para cada posição t, nº de valores distintos em codes[max(0, t-span) : t+1]."""
    n = len(codes)
    out = np.empty(n, dtype=np.int64)
    counts = Counter()
    window = deque()
    for t in range(n):
        counts[codes[t]] += 1
        window.append(codes[t])
        if len(window) > span + 1:
            old = window.popleft()
            counts[old] -= 1
            if counts[old] == 0:
                del counts[old]
        out[t] = len(counts)
    return out


def engineer_features(df):
    """
    Versão vetorizada de featmap_gen (AI4Risk/antifraud). Exige Time == posição da linha (vale no S-FFSD),
    então a janela [t - L, t] do original é a faixa de linhas max(0, t - L) .. t.
    Devolve um DataFrame com as mesmas colunas e a mesma ordem do original (colunas originais + 120 novas).
    """
    assert (df['Time'].to_numpy() == np.arange(len(df))).all(), 'Time deve ser 0..N-1 em ordem'
    amount = df['Amount'].astype('float64')
    cats = {c: pd.factorize(df[c])[0] for c in ('Target', 'Location', 'Type')}
    new = {}
    for span in TIME_SPAN:
        roll = amount.rolling(span + 1, min_periods=1)
        mean = roll.mean()
        new[f'trans_at_avg_{span}'] = mean
        new[f'trans_at_totl_{span}'] = roll.sum()
        new[f'trans_at_std_{span}'] = roll.std().fillna(0.0)          # std de 1 elemento é NaN no pandas; o original troca NaN por 0
        new[f'trans_at_bias_{span}'] = amount - mean
        new[f'trans_at_num_{span}'] = roll.count()
        new[f'trans_target_num_{span}'] = window_distinct(cats['Target'], span)
        new[f'trans_location_num_{span}'] = window_distinct(cats['Location'], span)
        new[f'trans_type_num_{span}'] = window_distinct(cats['Type'], span)
    return pd.concat([df.reset_index(drop=True), pd.DataFrame(new)], axis=1)


def signed_log1p(x):
    return np.sign(x) * np.log1p(np.abs(x))


def build_feature_matrix(engineered, include_ids=False):
    """
    Matriz de atributos para o PC-GNN: Amount + 120 atributos de janela (+ Time e códigos de Source/Target/Location/Type
    se include_ids). Log com sinal para as caudas pesadas e min-max por coluna, deixando tudo em [0, 1], como o
    YelpChi/Amazon, que é o que a normalização por linha do código original espera.
    """
    drop = ['Labels'] if include_ids else ['Labels', 'Time', 'Source', 'Target', 'Location', 'Type']
    feats = engineered.drop(columns=drop).copy()
    for col in ('Source', 'Target', 'Location', 'Type'):
        if col in feats:
            feats[col] = pd.factorize(feats[col])[0]
    x = signed_log1p(feats.to_numpy(dtype='float64'))
    low, high = x.min(axis=0), x.max(axis=0)
    span = np.where(high > low, high - low, 1.0)
    return ((x - low) / span).astype('float32'), list(feats.columns)


def relation_matrices(df):
    """
    Uma matriz esparsa por relação, igual ao grafo do antifraud: cada transação liga-se às 3 seguintes
    (ordenadas por Time) do mesmo grupo de Source/Target/Location/Type. Simetrizada e com laços próprios
    quando passa por sparse_to_adjlist.
    """
    n = len(df)
    out = {}
    for column in RELATIONS:
        codes = pd.factorize(df[column])[0]
        order = np.argsort(codes, kind='stable')              # dentro do grupo, mantém a ordem de Time
        sorted_codes = codes[order]
        src, dst = [], []
        for step in range(1, EDGES_PER_TRANS):
            same = sorted_codes[:-step] == sorted_codes[step:]
            src.append(order[:-step][same])
            dst.append(order[step:][same])
        src, dst = np.concatenate(src), np.concatenate(dst)
        mat = sp.coo_matrix((np.ones(len(src), dtype=np.int8), (src, dst)), shape=(n, n)).tocsr()
        mat = ((mat + mat.T) > 0).astype(np.int8)
        out[column.lower()] = mat
    return out


def export_for_pcgnn(df, engineered, data_dir, repo_src, include_ids=False):
    """Grava features, rótulos e listas de adjacência no formato que o código adaptado lê. Usa sparse_to_adjlist do repositório."""
    import json
    import sys
    from pathlib import Path
    data_dir = Path(data_dir)
    data_dir.mkdir(parents=True, exist_ok=True)
    sys.path.insert(0, str(Path(repo_src) / 'src'))
    from utils import sparse_to_adjlist
    X, names = build_feature_matrix(engineered, include_ids=include_ids)
    np.save(data_dir / 'sffsd_features.npy', X)
    np.save(data_dir / 'sffsd_labels.npy', df['Labels'].to_numpy(dtype='int64'))
    rels = relation_matrices(df)
    for name, mat in rels.items():
        sparse_to_adjlist(mat, str(data_dir / f'sffsd_{name}_adjlists.pickle'))
    homo = sum(rels.values())
    sparse_to_adjlist((homo > 0).astype(np.int8), str(data_dir / 'sffsd_homo_adjlists.pickle'))
    (data_dir / 'sffsd_manifest.json').write_text(json.dumps({'relations': list(rels), 'features': names}), encoding='utf-8')
    return rels, names

Writing /content/sffsd_support/sffsd_data.py


In [ ]:
if not (REPO / 'main.py').is_file():
    if REPO.exists() and any(REPO.iterdir()):
        raise RuntimeError(f'{REPO} existe mas não parece ser o clone do PC-GNN. Nada foi apagado; confira REPO.')
    subprocess.run(['git', 'clone', REPO_URL, str(REPO)], check=True)


def git(*args):
    return subprocess.run(['git', *args], cwd=REPO, capture_output=True, text=True, check=True).stdout.strip()


git('checkout', '--quiet', REPO_COMMIT)
COMMIT = git('rev-parse', 'HEAD')
assert COMMIT == REPO_COMMIT, f'commit inesperado: {COMMIT}'
LOCAL_CHANGES = git('status', '--porcelain', '--', 'main.py', 'src', 'config')
assert not LOCAL_CHANGES, f'o clone tem alterações locais: {LOCAL_CHANGES}'

sys.path.insert(0, str(SUPPORT))
import importlib
import sffsd_patch, sffsd_data
importlib.reload(sffsd_patch); importlib.reload(sffsd_data)
from sffsd_data import engineer_features, build_feature_matrix, relation_matrices, export_for_pcgnn

sffsd_patch.apply_patch(REPO, PATCHED)
diff = subprocess.run(['diff', '-ru', str(REPO / 'src'), str(PATCHED / 'src')], capture_output=True, text=True).stdout
(RESULTS_ROOT / 'patch.diff').write_text(diff, encoding='utf-8')
added = sum(1 for line in diff.splitlines() if line.startswith('+') and not line.startswith('+++'))
removed = sum(1 for line in diff.splitlines() if line.startswith('-') and not line.startswith('---'))
PATCH_SHA = hashlib.sha256(diff.encode('utf-8')).hexdigest()
print('Commit:', COMMIT)
print(f'Cópia adaptada em {PATCHED}: +{added} / -{removed} linhas em src/ (diff em {RESULTS_ROOT / "patch.diff"})')

Commit: 9d7d7fae491081178b6e12e193fb89cfc330f2be
Cópia adaptada em /content/PC-GNN-sffsd: +72 / -52 linhas em src/ (diff em /content/drive/MyDrive/PC_GNN_sffsd/patch.diff)


## 4. Dados: S-FFSD
Procura `S-FFSD.csv` (ou `.zip`) em `DATA_DRIVE_DIR`; se não achar, baixa de `AI4Risk/antifraud`. O SHA-256 do arquivo usado é conferido com o do arquivo que a equipe examinou ao preparar este notebook (um aviso, não um erro, se diferir).
Colunas: `Time` (0 a N−1, em ordem), `Source`, `Target`, `Amount`, `Location`, `Type`, `Labels` (0 benigno, 1 fraude, 2 sem rótulo).

In [ ]:
def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as source:
        for block in iter(lambda: source.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()


RAW = WORK / 'sffsd_raw'
RAW.mkdir(parents=True, exist_ok=True)
csv_path = RAW / 'S-FFSD.csv'


def extract_csv(zip_path):
    with ZipFile(zip_path) as archive:
        members = [n for n in archive.namelist() if Path(n).name.lower() == 's-ffsd.csv']
        if len(members) != 1:
            raise ValueError(f'{zip_path} deve conter exatamente um S-FFSD.csv.')
        with archive.open(members[0]) as source, csv_path.open('wb') as out:
            shutil.copyfileobj(source, out)


source_used = None
if DATA_DRIVE_DIR is not None and (DATA_DRIVE_DIR / 'S-FFSD.csv').is_file():
    shutil.copyfile(DATA_DRIVE_DIR / 'S-FFSD.csv', csv_path); source_used = 'Drive (.csv)'
elif DATA_DRIVE_DIR is not None and (DATA_DRIVE_DIR / 'S-FFSD.zip').is_file():
    extract_csv(DATA_DRIVE_DIR / 'S-FFSD.zip'); source_used = 'Drive (.zip)'
elif not csv_path.is_file():
    print('Baixando S-FFSD de AI4Risk/antifraud...')
    urlretrieve(SFFSD_URL, RAW / 'S-FFSD.zip')
    extract_csv(RAW / 'S-FFSD.zip'); source_used = 'download (AI4Risk/antifraud, ramo main)'
else:
    source_used = 'cópia desta sessão'

SFFSD_SHA = sha256_file(csv_path)
print('Fonte:', source_used)
print('SHA-256:', SFFSD_SHA, '(confere com o esperado)' if SFFSD_SHA == SFFSD_CSV_SHA256 else '<<< DIFERE do esperado; as estatísticas abaixo podem mudar')

df = pd.read_csv(csv_path)
assert list(df.columns) == ['Time', 'Source', 'Target', 'Amount', 'Location', 'Type', 'Labels'], df.columns.tolist()
assert (df['Time'].to_numpy() == np.arange(len(df))).all()
labels_all = df['Labels'].to_numpy(dtype='int64')
labeled_mask = labels_all < 2
n_labeled, n_fraud = int(labeled_mask.sum()), int((labels_all == 1).sum())
DATASET_STATS = {
    'nós (transações)': len(df), 'nós rotulados': n_labeled, 'nós sem rótulo': int((~labeled_mask).sum()),
    'fraudes (entre os rotulados)': n_fraud, '% de fraudes (rotulados)': round(100 * n_fraud / n_labeled, 2),
    'IR entre os rotulados': round((n_labeled - n_fraud) / n_fraud, 2),
    'Source distintos': int(df['Source'].nunique()), 'Target distintos': int(df['Target'].nunique()),
    'Location distintos': int(df['Location'].nunique()), 'Type distintos': int(df['Type'].nunique())}
display(pd.Series(DATASET_STATS, name='S-FFSD').to_frame())
display(df.groupby('Labels')['Amount'].describe().round(2))

Baixando S-FFSD de AI4Risk/antifraud...
Fonte: download (AI4Risk/antifraud, ramo main)
SHA-256: a2d78b983dfacaae394e4d69ece46ae880fd6bd4761bc5ee759977ba12b185c6 (confere com o esperado)


,S-FFSD
nós (transações),77881.00
nós rotulados,29643.00
nós sem rótulo,48238.00
fraudes (entre os rotulados),5256.00
% de fraudes (rotulados),17.73
IR entre os rotulados,4.64
Source distintos,30346.00
Target distintos,886.00
Location distintos,296.00
Type distintos,166.00


,count,mean,std,min,25%,50%,75%,max
Labels,,,,,,,,
0,24387.0,168.47,5172.14,0.0,0.09,15.25,63.89,800000.0
1,5256.0,169.20,2695.07,0.0,4.00,8.00,43.00,170000.0
2,48238.0,212.23,4525.69,0.0,7.00,18.60,71.60,648000.0


## 5. Atributos
O `antifraud` calcula, para cada transação, 8 estatísticas em 15 janelas de tempo (média, soma, desvio, viés e contagem do valor, e nº de `Target`, `Location` e `Type` distintos), com um laço sobre as linhas que leva **horas** (a verificação abaixo mede o tempo por linha). A versão vetorizada abaixo produz os mesmos números; ela é conferida contra uma **cópia literal da função original** nas primeiras linhas.
Como `Time` é igual à posição da linha, a janela `[t − L, t]` do original é a faixa de linhas `max(0, t − L)..t`.

In [ ]:
def featmap_gen_original(tmp_df):
    """Cópia literal de featmap_gen (AI4Risk/antifraud, feature_engineering/data_process.py), sem barra de progresso."""
    time_span = [2, 3, 5, 15, 20, 50, 100, 150, 200, 300, 864, 2590, 5100, 10000, 24000]
    time_name = [str(i) for i in time_span]
    time_list = tmp_df['Time']
    post_fe = []
    for trans_idx, trans_feat in tmp_df.iterrows():
        new_df = pd.Series(trans_feat)
        temp_time = new_df.Time
        temp_amt = new_df.Amount
        for length, tname in zip(time_span, time_name):
            lowbound = (time_list >= temp_time - length)
            upbound = (time_list <= temp_time)
            correct_data = tmp_df[lowbound & upbound]
            new_df['trans_at_avg_{}'.format(tname)] = correct_data['Amount'].mean()
            new_df['trans_at_totl_{}'.format(tname)] = correct_data['Amount'].sum()
            new_df['trans_at_std_{}'.format(tname)] = correct_data['Amount'].std()
            new_df['trans_at_bias_{}'.format(tname)] = temp_amt - correct_data['Amount'].mean()
            new_df['trans_at_num_{}'.format(tname)] = len(correct_data)
            new_df['trans_target_num_{}'.format(tname)] = len(correct_data.Target.unique())
            new_df['trans_location_num_{}'.format(tname)] = len(correct_data.Location.unique())
            new_df['trans_type_num_{}'.format(tname)] = len(correct_data.Type.unique())
        post_fe.append(new_df)
    return pd.DataFrame(post_fe)


VERIFY_ROWS = 40 if QUICK_TEST else 150
head = df.iloc[:VERIFY_ROWS].reset_index(drop=True)
t0 = time.time(); ref = featmap_gen_original(head).fillna(0); t_ref = time.time() - t0
t0 = time.time(); mine = engineer_features(head); t_mine = time.time() - t0
assert list(ref.columns) == list(mine.columns)
numeric = [c for c in ref.columns if c not in ('Source', 'Target', 'Location', 'Type')]
diff_max = float(np.abs(ref[numeric].to_numpy(float) - mine[numeric].to_numpy(float)).max())
assert diff_max < 1e-4, f'atributos vetorizados diferem do original: {diff_max}'
est_hours = t_ref / VERIFY_ROWS * len(df) / 3600
print(f'Verificação em {VERIFY_ROWS} linhas: diferença máxima {diff_max:.1e} | original {t_ref:.0f} s, vetorizada {t_mine:.2f} s '
      f'(o original levaria ~{est_hours:.1f} h para as {len(df)} linhas, mais por causa do custo crescente das janelas)')

t0 = time.time()
engineered = engineer_features(df)
print(f'Atributos de todas as linhas em {time.time() - t0:.1f} s: {engineered.shape[1] - 1} colunas (sem Labels)')

Verificação em 150 linhas: diferença máxima 2.5e-09 | original 10 s, vetorizada 0.02 s (o original levaria ~1.4 h para as 77881 linhas, mais por causa do custo crescente das janelas)
Atributos de todas as linhas em 3.2 s: 126 colunas (sem Labels)


## 6. Grafo e arquivos de entrada do PC-GNN
Gera as matrizes de atributos, os rótulos e as listas de adjacência (com a função `sparse_to_adjlist` do próprio repositório) na pasta `DATA_DIR`. A tabela mostra, por relação, o número de arestas e a **homofilia de rótulo** entre nós rotulados: quanto mais a taxa de vizinhos fraudulentos de um fraudador supera a de um benigno, mais a relação carrega sinal para o PC-GNN. Se as duas taxas forem parecidas, a agregação de vizinhança não ajuda (é o problema da camuflagem do artigo).

In [ ]:
STAMP = DATA_DIR / 'stamp.json'
stamp_now = {'csv_sha256': SFFSD_SHA, 'include_ids': INCLUDE_ID_FEATURES, 'patch_sha256': PATCH_SHA}
t0 = time.time()
rels = relation_matrices(df)
if STAMP.is_file() and json.loads(STAMP.read_text()) == stamp_now and (DATA_DIR / 'sffsd_manifest.json').is_file():
    manifest = json.loads((DATA_DIR / 'sffsd_manifest.json').read_text())
    FEATURE_NAMES = manifest['features']
    print('Arquivos de entrada já gerados para estes dados.')
else:
    _, FEATURE_NAMES = export_for_pcgnn(df, engineered, DATA_DIR, PATCHED, include_ids=INCLUDE_ID_FEATURES)
    STAMP.write_text(json.dumps(stamp_now))
    print(f'Arquivos de entrada gerados em {time.time() - t0:.0f} s.')
FEATURE_DIM = len(FEATURE_NAMES)

rows = []
for name, mat in rels.items():
    coo = sp.triu(mat, k=1).tocoo()
    both = labeled_mask[coo.row] & labeled_mask[coo.col]
    full = mat.tocoo()
    full_ok = (full.row != full.col) & labeled_mask[full.row] & labeled_mask[full.col]
    src_lab, dst_lab = labels_all[full.row[full_ok]], labels_all[full.col[full_ok]]
    rows.append({'relação': name, 'arestas': int(coo.nnz), 'grau médio': round(mat.sum() / len(df), 2),
                 'arestas entre rotulados': int(both.sum()),
                 'homofilia de rótulo': round(float((labels_all[coo.row[both]] == labels_all[coo.col[both]]).mean()), 3),
                 'P(vizinho fraude | fraude)': round(float((dst_lab[src_lab == 1] == 1).mean()), 3),
                 'P(vizinho fraude | benigno)': round(float((dst_lab[src_lab == 0] == 1).mean()), 3)})
GRAPH_STATS = pd.DataFrame(rows)
display(GRAPH_STATS)
print('Referência: fração de fraudes entre os rotulados =', round(n_fraud / n_labeled, 3), '| dimensão dos atributos =', FEATURE_DIM)

Arquivos de entrada gerados em 10 s.


,relação,arestas,grau médio,arestas entre rotulados,homofilia de rótulo,P(vizinho fraude | fraude),P(vizinho fraude | benigno)
0,source,85834,2.20,46863,0.829,0.250,0.097
1,target,153337,3.94,43639,0.843,0.592,0.097
2,location,154990,3.98,38535,0.833,0.598,0.105
3,type,155283,3.99,41665,0.840,0.595,0.099


Referência: fração de fraudes entre os rotulados = 0.177 | dimensão dos atributos = 121


## 7. Divisão, métricas e modelos de referência sem grafo
A divisão é refeita aqui com os mesmos comandos do código dos autores, para obter os índices de treino, validação e teste. O primeiro treino confere os tamanhos com o que o `main.py` imprime.

**Métricas** (as do artigo e do relatório): AUC, F1-macro, G-Mean, F1 de cada classe; acrescento a **AP** (precisão média), mais informativa que a AUC com fraude de ~18%. A AUC e a AP não dependem do limiar. F1 e G-Mean dependem; por isso há duas versões: com o limiar do arquivo de configuração (**τ = 0,5**) e com o **τ escolhido na validação** (maior F1-macro na grade 0,01 a 0,99).

**Modelos de referência sem grafo**: `HistGradientBoosting` e regressão logística (com `class_weight='balanced'`) nos mesmos atributos e na mesma divisão. Mostram quanto do desempenho vem dos atributos e quanto vem do grafo.

In [ ]:
labeled = np.where(labeled_mask)[0]
idx_train, idx_rest, y_train, y_rest = train_test_split(labeled.tolist(), labels_all[labeled], stratify=labels_all[labeled],
                                                        train_size=0.4, random_state=2, shuffle=True)
idx_valid, idx_test, y_valid, y_test = train_test_split(idx_rest, y_rest, stratify=y_rest, test_size=0.67, random_state=2, shuffle=True)
y_train, y_valid, y_test = map(np.asarray, (y_train, y_valid, y_test))
SPLIT = {'treino': (len(y_train), int(y_train.sum())), 'validação': (len(y_valid), int(y_valid.sum())), 'teste': (len(y_test), int(y_test.sum()))}
display(pd.DataFrame(SPLIT, index=['nós', 'fraudes']).T.assign(**{'% fraude': lambda t: (100 * t['fraudes'] / t['nós']).round(2)}))

# O código do autor quebra se o último lote de validação, teste ou treino tiver tamanho 0 (nº de nós múltiplo do lote)
N_SAMPLED = 2 * int(y_train.sum())
BATCH_SIZES_OK = [b for b in (128, 256, 512, 1024) if all(n % b for n in (len(y_valid), len(y_test), N_SAMPLED))]
print('Nós sorteados pelo Pick por época:', N_SAMPLED, '| tamanhos de lote seguros:', BATCH_SIZES_OK)

GRID = np.round(np.linspace(0.01, 0.99, 99), 2)


def metrics_at(y, p, thres):
    y, p = np.asarray(y), np.asarray(p)
    pred = (p >= thres).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    tpr = tp / (tp + fn) if tp + fn else 0.0
    tnr = tn / (tn + fp) if tn + fp else 0.0
    return {'f1_macro': float(f1_score(y, pred, average='macro', zero_division=0)),
            'f1_fraud': float(f1_score(y, pred, pos_label=1, zero_division=0)),
            'f1_benign': float(f1_score(y, pred, pos_label=0, zero_division=0)),
            'g_mean': float(math.sqrt(tpr * tnr)), 'tp': int(tp), 'tn': int(tn), 'fn': int(fn), 'fp': int(fp)}


def rank_metrics(y, p):
    return {'auc': float(roc_auc_score(y, p)), 'ap': float(average_precision_score(y, p))}


def best_threshold(y, p):
    """τ com maior F1-macro no conjunto dado (a validação); empate: o mais próximo de 0,5."""
    scores = [(metrics_at(y, p, t)['f1_macro'], -abs(t - 0.5), t) for t in GRID]
    return float(max(scores)[2])


def full_report(yv, pv, yt, pt, thres):
    tau = best_threshold(yv, pv)
    return {'tau': tau,
            'val': {**rank_metrics(yv, pv), **metrics_at(yv, pv, thres)},
            'test': {**rank_metrics(yt, pt), **metrics_at(yt, pt, thres)},
            'val_tuned': metrics_at(yv, pv, tau), 'test_tuned': metrics_at(yt, pt, tau)}


# ---- referências sem grafo ----
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
X_all = np.load(DATA_DIR / 'sffsd_features.npy')
REFERENCE = {}
for name, model in (('HistGradientBoosting (sem grafo)', HistGradientBoostingClassifier(max_iter=300, learning_rate=0.1, random_state=0)),
                    ('Regressão logística (sem grafo)', LogisticRegression(max_iter=2000, class_weight='balanced'))):
    t0 = time.time()
    model.fit(X_all[idx_train], y_train)
    pv, pt = model.predict_proba(X_all[idx_valid])[:, 1], model.predict_proba(X_all[idx_test])[:, 1]
    REFERENCE[name] = full_report(y_valid, pv, y_test, pt, 0.5)
    print(f'{name}: AUC teste {REFERENCE[name]["test"]["auc"]:.4f} | AP {REFERENCE[name]["test"]["ap"]:.4f} | {time.time() - t0:.0f} s')

,nós,fraudes,% fraude
treino,11857,2102,17.73
validação,5869,1041,17.74
teste,11917,2113,17.73


Nós sorteados pelo Pick por época: 4204 | tamanhos de lote seguros: [128, 256, 512, 1024]
HistGradientBoosting (sem grafo): AUC teste 0.8618 | AP 0.7040 | 2 s
Regressão logística (sem grafo): AUC teste 0.8143 | AP 0.6393 | 2 s


## 8. Executor
`main.py` não tem `--seed`: a semente vai no YAML da execução. Cada execução tem pasta própria em `RESULTS_ROOT/<etapa>/<configuração>/seed_<n>/` com `config.yml`, `train.log`, `probs.npz` (rótulos e probabilidades de validação e teste do **melhor checkpoint**, escolhido pela AUC de validação, como no código original) e `result.json`.

As métricas são recalculadas a partir de `probs.npz` e **conferidas com a linha de teste que o `main.py` imprime** (tolerância 1e-4). Se o `result.json` já existe com a mesma configuração e os mesmos dados, a execução é pulada. Se a configuração mudou, o notebook para em vez de misturar resultados.

In [ ]:
BASE_YAML = yaml.safe_load((REPO / 'config' / 'pcgnn_yelpchi.yml').read_text(encoding='utf-8'))
DATA_ID = hashlib.sha256(json.dumps([SFFSD_SHA, INCLUDE_ID_FEATURES, PATCH_SHA]).encode()).hexdigest()[:16]
THREADS = max(1, (os.cpu_count() or 1) // max(1, N_PARALLEL))
SPLIT_LINE = re.compile(r'Run on sffsd, postive/total num: \d+/\d+, train num (\d+),valid num (\d+), test num (\d+), test positive num (\d+)')
TEST_LINE = re.compile(r'GNN F1-binary-1:\s*([0-9.eE+-]+)\s+F1-binary-0:\s*([0-9.eE+-]+)\s+F1-macro:\s*([0-9.eE+-]+)\s+G-Mean:\s*([0-9.eE+-]+)\s+AUC:\s*([0-9.eE+-]+)')


def make_config(params, seed, run_dir):
    cfg = dict(BASE_YAML)                                   # YAML do YelpChi dos autores, sem alterações
    cfg.update(data_name='sffsd', data_dir=str(DATA_DIR) + '/', save_dir=str(run_dir / 'checkpoints') + '/', seed=seed,
               no_cuda=True, dump_probs=str(run_dir / 'probs.npz'), feat_norm='row', keep_ratio=0.5)
    cfg.update(params)
    if EPOCHS_OVERRIDE is not None:
        cfg['num_epochs'] = EPOCHS_OVERRIDE
    if any(isinstance(v, list) for v in cfg.values()):
        raise ValueError('O YAML tem hiperparâmetros em lista.')
    return cfg


def semantic(cfg):
    return {k: v for k, v in cfg.items() if k not in ('save_dir', 'data_dir', 'dump_probs')}


def write_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + '.tmp')
    tmp.write_text(json.dumps(payload, indent=2, ensure_ascii=False, allow_nan=False), encoding='utf-8')
    tmp.replace(path)


def run_dir_of(stage, tag, seed):
    return RESULTS_ROOT / stage / tag / f'seed_{seed}'


def run_one(stage, tag, params, seed):
    run_dir = run_dir_of(stage, tag, seed)
    cfg = make_config(params, seed, run_dir)
    sem = semantic(cfg)
    result_path = run_dir / 'result.json'
    if result_path.is_file():
        old = json.loads(result_path.read_text(encoding='utf-8'))
        if old.get('config') == sem and old.get('data_id') == DATA_ID:
            return old, True
        raise RuntimeError(f'{result_path} foi gerado com outra configuração ou outros dados. Use outro RESULTS_ROOT ou apague a pasta.')
    run_dir.mkdir(parents=True, exist_ok=True)
    cfg_path = run_dir / 'config.yml'
    cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=True), encoding='utf-8')
    env = dict(os.environ, OMP_NUM_THREADS=str(THREADS), MKL_NUM_THREADS=str(THREADS), PYTHONUNBUFFERED='1')
    t0 = time.time()
    with (run_dir / 'train.log').open('w', encoding='utf-8') as log:
        proc = subprocess.run([sys.executable, 'main.py', '--config', str(cfg_path)], cwd=PATCHED, stdout=log, stderr=subprocess.STDOUT, env=env)
    wall = time.time() - t0
    text = (run_dir / 'train.log').read_text(encoding='utf-8', errors='replace')
    if proc.returncode != 0:
        raise RuntimeError(f'{stage}/{tag}/seed_{seed} falhou (código {proc.returncode}). Fim do log:\n' + text[-1500:])

    # ---- extração e conferência ----
    restored = re.findall(r'^Restore model from epoch (\d+)', text, re.M)
    split = SPLIT_LINE.search(text)
    dims = re.search(r'^Feature dimension:\s*(\d+)', text, re.M)
    test_section = text.split('Model path:', 1)[1] if 'Model path:' in text else ''
    line = TEST_LINE.findall(test_section)
    if len(restored) != 1 or split is None or len(line) != 1 or dims is None:
        raise RuntimeError(f'Log incompleto em {run_dir / "train.log"}')
    if tuple(map(int, split.groups())) != (len(y_train), len(y_valid), len(y_test), int(y_test.sum())):
        raise RuntimeError(f'Divisão do main.py difere da calculada no notebook: {split.groups()}')
    if int(dims.group(1)) != FEATURE_DIM:
        raise RuntimeError(f'Dimensão de atributos {dims.group(1)} != {FEATURE_DIM}')
    probs = np.load(run_dir / 'probs.npz')
    yv, pv, yt, pt = probs['valid_y'], probs['valid_p'], probs['test_y'], probs['test_p']
    if not (np.array_equal(yv, y_valid) and np.array_equal(yt, y_test)):
        raise RuntimeError('Rótulos gravados diferem da divisão calculada.')
    report = full_report(yv, pv, yt, pt, cfg['thres'])
    f1_fraud, f1_benign, f1_macro, g_mean, auc = map(float, line[0])
    for key, value in (('f1_fraud', f1_fraud), ('f1_benign', f1_benign), ('f1_macro', f1_macro), ('g_mean', g_mean), ('auc', auc)):
        if abs(report['test'][key] - value) > 1e-4:
            raise RuntimeError(f'Métrica {key}: recalculada {report["test"][key]:.4f} != log {value:.4f}')
    epoch_times = [float(x) for x in re.findall(r'Epoch: \d+, loss: \S+, time: ([0-9.]+)s', text)]
    result = {'stage': stage, 'tag': tag, 'seed': seed, 'params': params, 'config': sem, 'data_id': DATA_ID,
              'best_epoch': int(restored[0]), 'train_seconds': sum(epoch_times), 'wall_seconds': wall,
              'finished_utc': datetime.now(timezone.utc).isoformat(), **report}
    write_json(result_path, result)
    return result, False


def run_jobs(jobs, label):
    """jobs: lista de (stage, tag, params, seed). Roda com N_PARALLEL execuções simultâneas e devolve os resultados na mesma ordem."""
    results = [None] * len(jobs)
    pending = [i for i, j in enumerate(jobs)]
    done_count, start = 0, time.time()
    new_runs = 0
    with ThreadPoolExecutor(max_workers=N_PARALLEL) as pool:
        futures = {pool.submit(run_one, *jobs[i]): i for i in pending}
        for future in as_completed(futures):
            i = futures[future]
            result, cached = future.result()
            results[i] = result
            done_count += 1
            new_runs += (not cached)
            elapsed = (time.time() - start) / 60
            remaining = (len(jobs) - done_count) * elapsed / max(new_runs, 1) if new_runs else 0
            t = result['test']
            print(f'[{label}] {done_count}/{len(jobs)} {result["tag"]} seed {result["seed"]}{" (já pronta)" if cached else ""}: '
                  f'val AUC {result["val"]["auc"]:.4f} | teste AUC {t["auc"]:.4f} F1-macro {t["f1_macro"]:.4f} G-Mean {t["g_mean"]:.4f} '
                  f'| época {result["best_epoch"]} | {result["wall_seconds"] / 60:.1f} min'
                  f'{f" | faltam ~{remaining:.0f} min" if new_runs and done_count < len(jobs) else ""}', flush=True)
    return results


def to_frame(results):
    rows = []
    for r in results:
        row = {'tag': r['tag'], 'seed': r['seed'], 'best_epoch': r['best_epoch'], 'train_seconds': round(r['train_seconds'], 1), 'tau': r['tau']}
        for split_key in ('val', 'test', 'val_tuned', 'test_tuned'):
            for metric, value in r[split_key].items():
                if metric in ('tp', 'tn', 'fn', 'fp'):
                    continue
                row[f'{split_key}_{metric}'] = value
        rows.append(row)
    return pd.DataFrame(rows)


print('Executor pronto. DATA_ID =', DATA_ID, '| threads por execução:', THREADS)

Executor pronto. DATA_ID = 7439d832b568023f | threads por execução: 1


## 9. Linha de base: PC-GNN com a configuração dos autores
YAML do YelpChi **sem alterações** (lr 0,01; weight decay 0,001; lote 1024; ρ 0,5; α 2; 51 épocas; τ 0,5; normalização por linha; metade dos vizinhos mantida), com as 10 sementes finais. É o que "rodar o código" produziria no S-FFSD.

In [ ]:
BASE_PARAMS = {}     # nada alterado em relação ao YAML
t0 = time.time()
base_results = run_jobs([('final', 'base', BASE_PARAMS, s) for s in SEEDS], 'padrão')
df_base = to_frame(base_results)
print(f'\nLinha de base concluída em {(time.time() - t0) / 60:.1f} min nesta sessão.')
display(df_base[['seed', 'best_epoch', 'val_auc', 'test_auc', 'test_ap', 'test_f1_macro', 'test_g_mean', 'test_f1_fraud', 'tau', 'test_tuned_f1_macro', 'test_tuned_g_mean']].round(4))

[padrão] 1/10 base seed 42: val AUC 0.7552 | teste AUC 0.7671 F1-macro 0.7151 G-Mean 0.6421 | época 50 | 8.4 min | faltam ~76 min
[padrão] 2/10 base seed 448: val AUC 0.7526 | teste AUC 0.7639 F1-macro 0.6916 G-Mean 0.6660 | época 50 | 8.4 min | faltam ~34 min
[padrão] 3/10 base seed 854: val AUC 0.7541 | teste AUC 0.7659 F1-macro 0.7135 G-Mean 0.6471 | época 45 | 8.3 min | faltam ~39 min
[padrão] 4/10 base seed 29493: val AUC 0.7541 | teste AUC 0.7650 F1-macro 0.6397 G-Mean 0.6843 | época 35 | 8.3 min | faltam ~25 min
[padrão] 5/10 base seed 72: val AUC 0.7497 | teste AUC 0.7623 F1-macro 0.7100 G-Mean 0.6502 | época 30 | 8.1 min | faltam ~25 min
[padrão] 6/10 base seed 88867: val AUC 0.7534 | teste AUC 0.7642 F1-macro 0.6818 G-Mean 0.6740 | época 35 | 8.2 min | faltam ~17 min
[padrão] 7/10 base seed 2024: val AUC 0.7521 | teste AUC 0.7651 F1-macro 0.7163 G-Mean 0.6149 | época 20 | 8.4 min | faltam ~14 min
[padrão] 8/10 base seed 123: val AUC 0.7494 | teste AUC 0.7593 F1-macro 0.6916 G

,seed,best_epoch,val_auc,test_auc,test_ap,test_f1_macro,test_g_mean,test_f1_fraud,tau,test_tuned_f1_macro,test_tuned_g_mean
0,42,50,0.7552,0.7671,0.5962,0.7151,0.6421,0.5157,0.55,0.7221,0.6122
1,448,50,0.7526,0.7639,0.5978,0.6916,0.6660,0.4939,0.58,0.7299,0.6178
2,854,45,0.7541,0.7659,0.5966,0.7135,0.6471,0.5150,0.57,0.7254,0.6058
3,29493,35,0.7541,0.7650,0.5938,0.6397,0.6843,0.4526,0.61,0.7213,0.6120
4,88867,35,0.7534,0.7642,0.5948,0.6818,0.6740,0.4860,0.61,0.7242,0.6070
5,72,30,0.7497,0.7623,0.5920,0.7100,0.6502,0.5113,0.57,0.7222,0.6077
6,123,45,0.7494,0.7593,0.5901,0.6916,0.6645,0.4934,0.58,0.7245,0.6111
7,2024,20,0.7521,0.7651,0.5910,0.7163,0.6149,0.5096,0.51,0.7169,0.6086
8,31337,40,0.7495,0.7609,0.5953,0.4557,0.5756,0.3632,0.68,0.7232,0.6209
9,7,30,0.7491,0.7591,0.5919,0.5823,0.6720,0.4176,0.67,0.7219,0.6100


## 10. Busca de hiperparâmetros (somente validação)
**Protocolo.** (A) Sorteio de `N_RANDOM_CONFIGS` configurações num espaço em torno do YAML dos autores, cada uma com a(s) semente(s) de busca; (B) as `TOP_K` melhores pela **AUC de validação** são repetidas com sementes extras; (C) a melhor pela média da AUC de validação é executada com as 10 sementes finais. **O teste não participa de nenhuma escolha**: as colunas de teste da busca são gravadas, mas não aparecem nas tabelas desta seção. As sementes da busca são diferentes das finais.

A AUC de validação é o critério porque não depende do limiar; o limiar τ é escolhido depois, também na validação (seção 12). Hiperparâmetros: taxa de aprendizado, *weight decay*, tamanho do lote, ρ (fração de vizinhos fraudulentos acrescentados pelo *Choose*), α (peso da perda do preditor de rótulo), `keep_ratio` (fração de vizinhos mantida pela subamostragem do *Choose*, 0,5 no original; 1,0 desliga a subamostragem), dimensão do *embedding*, normalização por linha dos atributos e nº de épocas.

In [ ]:
SPACE = {'lr': [0.001, 0.003, 0.005, 0.01, 0.02],
         'weight_decay': [0.0, 1e-4, 5e-4, 1e-3, 5e-3],
         'batch_size': BATCH_SIZES_OK or [1024],
         'rho': [0.2, 0.5, 0.8],
         'alpha': [0.5, 1, 2, 4],
         'keep_ratio': [0.3, 0.5, 0.7, 1.0],
         'emb_size': [32, 64, 128],
         'feat_norm': ['row', 'none'],
         'num_epochs': [31, 51]}
rng = random.Random(2026)
search_params, seen = [], set()
while len(search_params) < N_RANDOM_CONFIGS:
    cand = {k: rng.choice(v) for k, v in SPACE.items()}
    key = json.dumps(cand, sort_keys=True)
    if key not in seen:
        seen.add(key)
        search_params.append(cand)
SEARCH = {f'cfg{i + 1:02d}': p for i, p in enumerate(search_params)}
display(pd.DataFrame(SEARCH).T)
json.dump({'space': SPACE, 'configs': SEARCH}, open(RESULTS_ROOT / 'search_space.json', 'w'), indent=2)

,lr,weight_decay,batch_size,rho,alpha,keep_ratio,emb_size,feat_norm,num_epochs
cfg01,0.001,0.0005,128,0.2,4,1.0,128,none,31
cfg02,0.001,0.005,128,0.2,2,0.3,64,row,51
cfg03,0.005,0.0001,1024,0.5,2,0.7,64,row,51
cfg04,0.001,0.005,512,0.5,4,0.5,128,none,31
cfg05,0.005,0.0005,1024,0.5,0.5,1.0,128,none,31
cfg06,0.01,0.005,1024,0.5,2,1.0,128,none,31
cfg07,0.02,0.0001,256,0.2,0.5,1.0,64,row,51
cfg08,0.003,0.0,1024,0.8,4,0.7,128,none,51
cfg09,0.02,0.005,1024,0.5,4,0.5,64,none,31
cfg10,0.001,0.0001,1024,0.8,4,0.7,32,none,51


In [14]:
t_search = time.time()
search_runs = []
jobs = [('search', tag, p, s) for tag, p in SEARCH.items() for s in TUNE_SEEDS]
if TUNE_TIME_BUDGET_MIN is None:
    search_runs = run_jobs(jobs, 'busca')
else:
    # em lotes de N_PARALLEL, parando quando o orçamento de tempo acabar (as configurações não concluídas ficam de fora)
    for i in range(0, len(jobs), N_PARALLEL):
        if (time.time() - t_search) / 60 > TUNE_TIME_BUDGET_MIN:
            print(f'Orçamento de {TUNE_TIME_BUDGET_MIN} min esgotado: {i} de {len(jobs)} execuções feitas.')
            break
        search_runs += run_jobs(jobs[i:i + N_PARALLEL], 'busca')


def val_table(results, group_by='tag'):
    d = to_frame(results)
    agg = d.groupby(group_by).agg(n=('seed', 'count'), val_auc=('val_auc', 'mean'), val_ap=('val_ap', 'mean'),
                                  val_f1_tuned=('val_tuned_f1_macro', 'mean'), epoca=('best_epoch', 'mean'),
                                  train_s=('train_seconds', 'mean')).round(4)
    return agg.sort_values('val_auc', ascending=False)


search_table = val_table(search_runs)
search_table = search_table.join(pd.DataFrame(SEARCH).T)
print(f'\nBusca concluída em {(time.time() - t_search) / 60:.1f} min. Ordenada pela AUC de validação (o teste não é mostrado):')
display(search_table)

[busca] 1/12 cfg01 seed 1001: val AUC 0.8174 | teste AUC 0.8350 F1-macro 0.6469 G-Mean 0.7323 | época 30 | 5.4 min | faltam ~60 min
[busca] 2/12 cfg02 seed 1001: val AUC 0.7352 | teste AUC 0.7471 F1-macro 0.7027 G-Mean 0.6431 | época 15 | 7.9 min | faltam ~40 min
[busca] 3/12 cfg04 seed 1001: val AUC 0.7762 | teste AUC 0.7868 F1-macro 0.6427 G-Mean 0.6958 | época 30 | 5.3 min | faltam ~40 min
[busca] 4/12 cfg03 seed 1001: val AUC 0.7789 | teste AUC 0.7939 F1-macro 0.7088 G-Mean 0.6772 | época 50 | 8.8 min | faltam ~28 min
[busca] 5/12 cfg05 seed 1001: val AUC 0.7761 | teste AUC 0.7885 F1-macro 0.7062 G-Mean 0.6779 | época 30 | 5.8 min | faltam ~27 min
[busca] 6/12 cfg06 seed 1001: val AUC 0.7641 | teste AUC 0.7755 F1-macro 0.6226 G-Mean 0.6907 | época 30 | 5.7 min | faltam ~20 min
[busca] 7/12 cfg07 seed 1001: val AUC 0.8008 | teste AUC 0.8159 F1-macro 0.7431 G-Mean 0.6707 | época 50 | 8.0 min | faltam ~19 min
[busca] 8/12 cfg08 seed 1001: val AUC 0.7861 | teste AUC 0.8007 F1-macro 0.3

,n,val_auc,val_ap,val_f1_tuned,epoca,train_s,lr,weight_decay,batch_size,rho,alpha,keep_ratio,emb_size,feat_norm,num_epochs
tag,,,,,,,,,,,,,,,
cfg01,1,0.8174,0.6425,0.7389,30.0,248.5,0.001,0.0005,128,0.2,4,1.0,128,none,31
cfg07,1,0.8008,0.6215,0.7292,50.0,385.6,0.02,0.0001,256,0.2,0.5,1.0,64,row,51
cfg08,1,0.7861,0.6055,0.7252,45.0,429.3,0.003,0.0,1024,0.8,4,0.7,128,none,51
cfg03,1,0.7789,0.5983,0.7232,50.0,423.2,0.005,0.0001,1024,0.5,2,0.7,64,row,51
cfg04,1,0.7762,0.5901,0.7226,30.0,244.7,0.001,0.005,512,0.5,4,0.5,128,none,31
cfg05,1,0.7761,0.5940,0.7253,30.0,262.9,0.005,0.0005,1024,0.5,0.5,1.0,128,none,31
cfg10,1,0.7752,0.5976,0.7245,50.0,436.3,0.001,0.0001,1024,0.8,4,0.7,32,none,51
cfg06,1,0.7641,0.5835,0.7170,30.0,262.3,0.01,0.005,1024,0.5,2,1.0,128,none,31
cfg12,1,0.7593,0.5800,0.7171,30.0,154.5,0.01,0.005,512,0.5,1,1.0,128,none,31


In [15]:
print(f'Para referência, o padrão teve AUC de validação média de {df_base["val_auc"].mean():.4f} nas sementes finais (outras sementes). '
      'A comparação justa entre padrão e ajustado é a da seção 12, nas mesmas sementes.')
top_tags = list(search_table.index[:TOP_K])
print('Candidatos à confirmação:', top_tags)
confirm_jobs = [('search', tag, SEARCH[tag], s) for tag in top_tags for s in TUNE_SEEDS + CONFIRM_EXTRA_SEEDS]
confirm_runs = run_jobs(confirm_jobs, 'confirmação')
confirm_table = val_table(confirm_runs).join(pd.DataFrame(SEARCH).T)
print('\nConfirmação (média da AUC de validação em todas as sementes de busca):')
display(confirm_table)
BEST_TAG = confirm_table.index[0]
BEST_PARAMS = SEARCH[BEST_TAG]
print('Configuração escolhida:', BEST_TAG, BEST_PARAMS)

# sensibilidade (indicativa: poucas configurações)
rows = []
sens = search_table.copy()
for hp in SPACE:
    if sens[hp].nunique() > 1:
        numeric = pd.to_numeric(sens[hp], errors='coerce')
        if numeric.notna().all():
            rho_s, p_s = spearmanr(numeric, sens['val_auc'])
            rows.append({'hiperparâmetro': hp, 'Spearman com a AUC de validação': round(float(rho_s), 2), 'p': round(float(p_s), 3)})
if rows:
    display(pd.DataFrame(rows))

Para referência, o padrão teve AUC de validação média de 0.7519 nas sementes finais (outras sementes). A comparação justa entre padrão e ajustado é a da seção 12, nas mesmas sementes.
Candidatos à confirmação: ['cfg01', 'cfg07', 'cfg08']
[confirmação] 1/9 cfg01 seed 1001 (já pronta): val AUC 0.8174 | teste AUC 0.8350 F1-macro 0.6469 G-Mean 0.7323 | época 30 | 5.4 min
[confirmação] 2/9 cfg01 seed 1003: val AUC 0.8078 | teste AUC 0.8243 F1-macro 0.7504 G-Mean 0.6371 | época 30 | 5.5 min | faltam ~38 min
[confirmação] 3/9 cfg07 seed 1001 (já pronta): val AUC 0.8008 | teste AUC 0.8159 F1-macro 0.7431 G-Mean 0.6707 | época 50 | 8.0 min | faltam ~33 min
[confirmação] 4/9 cfg01 seed 1002: val AUC 0.8094 | teste AUC 0.8221 F1-macro 0.7502 G-Mean 0.6409 | época 30 | 5.5 min | faltam ~14 min
[confirmação] 5/9 cfg07 seed 1003: val AUC 0.7902 | teste AUC 0.8073 F1-macro 0.7144 G-Mean 0.6864 | época 50 | 8.1 min | faltam ~18 min
[confirmação] 6/9 cfg08 seed 1001 (já pronta): val AUC 0.7861 | teste 

,n,val_auc,val_ap,val_f1_tuned,epoca,train_s,lr,weight_decay,batch_size,rho,alpha,keep_ratio,emb_size,feat_norm,num_epochs
tag,,,,,,,,,,,,,,,
cfg01,3,0.8115,0.6340,0.7378,30.0000,247.9667,0.001,0.0005,128,0.2,4,1.0,128,none,31
cfg07,3,0.7937,0.6107,0.7267,50.0000,389.3667,0.02,0.0001,256,0.2,0.5,1.0,64,row,51
cfg08,3,0.7811,0.6022,0.7260,46.6667,431.5000,0.003,0.0,1024,0.8,4,0.7,128,none,51


Configuração escolhida: cfg01 {'lr': 0.001, 'weight_decay': 0.0005, 'batch_size': 128, 'rho': 0.2, 'alpha': 4, 'keep_ratio': 1.0, 'emb_size': 128, 'feat_norm': 'none', 'num_epochs': 31}


,hiperparâmetro,Spearman com a AUC de validação,p
0,lr,-0.20,0.530
1,weight_decay,-0.72,0.008
2,batch_size,-0.18,0.584
3,rho,-0.07,0.827
4,alpha,0.15,0.631
5,keep_ratio,0.47,0.126
6,emb_size,0.31,0.325
7,num_epochs,0.22,0.491


## 11. Configuração ajustada nas 10 sementes finais
A configuração escolhida na seção 10 roda nas **mesmas 10 sementes** da linha de base, o que permite comparação pareada por semente.

In [ ]:
t0 = time.time()
tuned_results = run_jobs([('final', 'tuned', BEST_PARAMS, s) for s in SEEDS], 'ajustado')
df_tuned = to_frame(tuned_results)
print(f'\nConcluído em {(time.time() - t0) / 60:.1f} min nesta sessão. Parâmetros ajustados: {BEST_PARAMS}')
display(df_tuned[['seed', 'best_epoch', 'val_auc', 'test_auc', 'test_ap', 'test_f1_macro', 'test_g_mean', 'test_f1_fraud', 'tau', 'test_tuned_f1_macro', 'test_tuned_g_mean']].round(4))

## 12. Resultados no teste: efeito dos hiperparâmetros × efeito do limiar
Quatro variantes, nas mesmas 10 sementes: **padrão** e **ajustado**, cada um com **τ = 0,5** (o do arquivo de configuração) e com **τ da validação** (escolhido semente a semente, só com a validação). Média ± desvio padrão amostral (`ddof=1`). A AUC e a AP não dependem de τ.
O teste de Wilcoxon pareado serve só como indicação: dez pares e inicializações não idênticas.

In [17]:
METRICS = ['auc', 'ap', 'f1_macro', 'g_mean', 'f1_fraud', 'f1_benign']
LABELS = {'auc': 'AUC', 'ap': 'AP', 'f1_macro': 'F1-macro', 'g_mean': 'G-Mean', 'f1_fraud': 'F1-fraude', 'f1_benign': 'F1-benigno'}


def col(df, metric, tuned):
    if metric in ('auc', 'ap'):
        return df[f'test_{metric}']
    return df[f'test_tuned_{metric}'] if tuned else df[f'test_{metric}']


VARIANTS = [('PC-GNN padrão, τ = 0,5', df_base, False), ('PC-GNN padrão, τ da validação', df_base, True),
            ('PC-GNN ajustado, τ = 0,5', df_tuned, False), ('PC-GNN ajustado, τ da validação', df_tuned, True)]
rows = []
for name, d, tuned in VARIANTS:
    row = {'variante': name}
    for m in METRICS:
        v = col(d, m, tuned)
        row[LABELS[m]] = f'{v.mean():.4f} ± {v.std(ddof=1):.4f}'
    rows.append(row)
for name, rep in REFERENCE.items():
    rows.append({'variante': name + ', τ da validação', **{LABELS[m]: f'{(rep["test"] if m in ("auc", "ap") else rep["test_tuned"])[m]:.4f}' for m in METRICS}})
SUMMARY = pd.DataFrame(rows).set_index('variante')
display(SUMMARY)

# comparação pareada por semente: ajustado − padrão
pair = []
a, b = df_base.set_index('seed'), df_tuned.set_index('seed')
for label, tuned in (('τ = 0,5', False), ('τ da validação', True)):
    for m in METRICS:
        x, y = col(a, m, tuned).loc[SEEDS].to_numpy(), col(b, m, tuned).loc[SEEDS].to_numpy()
        try:
            p_value = float(wilcoxon(y, x).pvalue)
        except ValueError:
            p_value = float('nan')
        pair.append({'limiar': label, 'métrica': LABELS[m], 'padrão': round(x.mean(), 4), 'ajustado': round(y.mean(), 4),
                     'Δ (ajustado − padrão)': round(float((y - x).mean()), 4), 'sementes em que ajustado > padrão': int((y > x).sum()),
                     'p (Wilcoxon)': round(p_value, 4)})
PAIRED = pd.DataFrame(pair)
display(PAIRED)

# efeito isolado do limiar dentro de cada configuração (τ da validação − τ = 0,5), mesmas sementes
eff = []
for name, d in (('padrão', df_base), ('ajustado', df_tuned)):
    for m in ('f1_macro', 'g_mean', 'f1_fraud'):
        delta = d[f'test_tuned_{m}'] - d[f'test_{m}']
        eff.append({'configuração': name, 'métrica': LABELS[m], 'Δ por usar τ da validação': round(float(delta.mean()), 4),
                    'desvio entre sementes (τ = 0,5)': round(float(d[f'test_{m}'].std(ddof=1)), 4),
                    'desvio entre sementes (τ validação)': round(float(d[f'test_tuned_{m}'].std(ddof=1)), 4)})
display(pd.DataFrame(eff))
print('τ escolhidos na validação  padrão:', df_base['tau'].round(2).tolist(), '| ajustado:', df_tuned['tau'].round(2).tolist())

,AUC,AP,F1-macro,G-Mean,F1-fraude,F1-benigno
variante,,,,,,
"PC-GNN padrão, τ = 0,5",0.7633 ± 0.0028,0.5940 ± 0.0026,0.6598 ± 0.0831,0.6491 ± 0.0325,0.4758 ± 0.0504,0.8437 ± 0.1166
"PC-GNN padrão, τ da validação",0.7633 ± 0.0028,0.5940 ± 0.0026,0.7232 ± 0.0033,0.6113 ± 0.0048,0.5185 ± 0.0054,0.9278 ± 0.0018
"PC-GNN ajustado, τ = 0,5",0.8295 ± 0.0037,0.6664 ± 0.0049,0.6790 ± 0.0961,0.6805 ± 0.0412,0.5193 ± 0.0655,0.8386 ± 0.1272
"PC-GNN ajustado, τ da validação",0.8295 ± 0.0037,0.6664 ± 0.0049,0.7512 ± 0.0035,0.6668 ± 0.0079,0.5730 ± 0.0059,0.9293 ± 0.0022
"HistGradientBoosting (sem grafo), τ da validação",0.8618,0.7040,0.7648,0.6839,0.5968,0.9327
"Regressão logística (sem grafo), τ da validação",0.8143,0.6393,0.7375,0.6588,0.5513,0.9236


,limiar,métrica,padrão,ajustado,Δ (ajustado − padrão),sementes em que ajustado > padrão,p (Wilcoxon)
0,"τ = 0,5",AUC,0.7633,0.8295,0.0662,10,0.0020
1,"τ = 0,5",AP,0.5940,0.6664,0.0725,10,0.0020
2,"τ = 0,5",F1-macro,0.6598,0.6790,0.0192,6,0.6250
3,"τ = 0,5",G-Mean,0.6491,0.6805,0.0314,7,0.1309
4,"τ = 0,5",F1-fraude,0.4758,0.5193,0.0435,7,0.2754
5,"τ = 0,5",F1-benigno,0.8437,0.8386,-0.0051,5,1.0000
6,τ da validação,AUC,0.7633,0.8295,0.0662,10,0.0020
7,τ da validação,AP,0.5940,0.6664,0.0725,10,0.0020
8,τ da validação,F1-macro,0.7232,0.7512,0.0280,10,0.0020
9,τ da validação,G-Mean,0.6113,0.6668,0.0555,10,0.0020


,configuração,métrica,Δ por usar τ da validação,"desvio entre sementes (τ = 0,5)",desvio entre sementes (τ validação)
0,padrão,F1-macro,0.0634,0.0831,0.0033
1,padrão,G-Mean,-0.0378,0.0325,0.0048
2,padrão,F1-fraude,0.0426,0.0504,0.0054
3,ajustado,F1-macro,0.0722,0.0961,0.0035
4,ajustado,G-Mean,-0.0137,0.0412,0.0079
5,ajustado,F1-fraude,0.0537,0.0655,0.0059


τ escolhidos na validação  padrão: [0.55, 0.58, 0.57, 0.61, 0.61, 0.57, 0.58, 0.51, 0.68, 0.67] | ajustado: [0.59, 0.47, 0.56, 0.47, 0.57, 0.64, 0.68, 0.66, 0.52, 0.44]


### Figura: cada semente, padrão × ajustado

In [ ]:
INK, MUTED, BLUE, ORANGE = '#1f2937', '#6b7280', '#2563eb', '#d97706'
fig, axes = plt.subplots(1, 3, figsize=(15.5, 4.6), constrained_layout=True)
shown = [('auc', 'AUC', False), ('f1_macro', 'F1-macro', True), ('g_mean', 'G-Mean', True)]
for axis, (m, title, tuned_split) in zip(axes, shown):
    for k, (name, d, tuned) in enumerate(VARIANTS if tuned_split else [VARIANTS[0], VARIANTS[2]]):
        values = col(d, m, tuned).to_numpy()
        color = BLUE if 'padrão' in name else ORANGE
        x = k + np.linspace(-0.12, 0.12, len(values))
        axis.scatter(x, values, s=30, color=color, alpha=0.85, edgecolor='white', linewidth=0.7, zorder=3)
        axis.errorbar(k, values.mean(), yerr=values.std(ddof=1) if len(values) > 1 else 0, color=INK, capsize=6, linewidth=2, zorder=4)
    labels = [v[0].replace('PC-GNN ', '').replace(', ', '\n') for v in (VARIANTS if tuned_split else [VARIANTS[0], VARIANTS[2]])]
    axis.set_xticks(range(len(labels))); axis.set_xticklabels(labels, fontsize=8, color=INK)
    axis.set_title(f'{title} no teste (n={len(SEEDS)} sementes)', loc='left', fontweight='bold', color=INK)
    axis.grid(axis='y', color='#e5e7eb', zorder=0); axis.spines[['top', 'right']].set_visible(False)
    axis.tick_params(colors=MUTED)
plt.show()

## 13. Custo computacional
Soma só o tempo dos lotes de treino impresso pelo `main.py` (sem validação nem teste). Com 2 execuções em paralelo e 1 *thread* cada, o tempo por execução é maior do que seria sozinha.

In [19]:
cost = pd.concat([df_base.assign(grupo='padrão'), df_tuned.assign(grupo='ajustado')]).groupby('grupo').agg(
    execuções=('seed', 'count'), treino_s=('train_seconds', 'mean'), época_média=('best_epoch', 'mean')).round(1)
display(cost)

,execuções,treino_s,época_média
grupo,,,
ajustado,10,251.8,29.5
padrão,10,400.9,38.0


## 14. Exportar e resumo para colar na conversa
Grava no Drive as tabelas, a figura e um manifesto (commit, versões, SHA-256 dos dados, `diff` da adaptação, espaço de busca, sementes). A última célula imprime um **resumo em texto**: copie a saída inteira e cole na conversa.

In [20]:
REPORT = RESULTS_ROOT / 'reports' / datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')
REPORT.mkdir(parents=True, exist_ok=False)
pd.concat([df_base.assign(grupo='padrao'), df_tuned.assign(grupo='ajustado')]).to_csv(REPORT / 'final_por_semente.csv', index=False)
search_table.to_csv(REPORT / 'busca_validacao.csv')
confirm_table.to_csv(REPORT / 'confirmacao_validacao.csv')
SUMMARY.to_csv(REPORT / 'resumo_teste.csv')
PAIRED.to_csv(REPORT / 'pareado_ajustado_vs_padrao.csv', index=False)
GRAPH_STATS.to_csv(REPORT / 'grafo.csv', index=False)
fig.savefig(REPORT / 'sementes_padrao_vs_ajustado.png', dpi=160)
write_json(REPORT / 'manifest.json', {
    'notebook_version': 'v4-sffsd', 'created_utc': datetime.now(timezone.utc).isoformat(), 'repo_url': REPO_URL, 'commit': COMMIT,
    'patch_sha256': PATCH_SHA, 'versions': VERSIONS, 'python': sys.version, 'platform': platform.platform(), 'cpus': os.cpu_count(),
    'sffsd_csv_sha256': SFFSD_SHA, 'sffsd_source': source_used, 'include_id_features': INCLUDE_ID_FEATURES, 'feature_dim': FEATURE_DIM,
    'dataset_stats': DATASET_STATS, 'split': {k: list(v) for k, v in SPLIT.items()},
    'seeds': SEEDS, 'tune_seeds': TUNE_SEEDS, 'confirm_extra_seeds': CONFIRM_EXTRA_SEEDS, 'n_random_configs': N_RANDOM_CONFIGS, 'top_k': TOP_K,
    'best_tag': BEST_TAG, 'best_params': BEST_PARAMS, 'base_yaml': BASE_YAML, 'epochs_override': EPOCHS_OVERRIDE, 'quick_test': QUICK_TEST})
print('Relatório salvo em:', REPORT)

Relatório salvo em: /content/drive/MyDrive/PC_GNN_sffsd/reports/20261117_230037


In [21]:
def md_table(frame, index=True):
    f = frame.reset_index() if index else frame
    head = '| ' + ' | '.join(map(str, f.columns)) + ' |\n|' + '---|' * len(f.columns) + '\n'
    return head + '\n'.join('| ' + ' | '.join(str(v) for v in row) + ' |' for row in f.itertuples(index=False))


lines = ['=== RESUMO S-FFSD (copie daqui até FIM) ===', '',
         f'Commit PC-GNN: {COMMIT} | adaptação +{added}/-{removed} linhas | S-FFSD sha256 {SFFSD_SHA[:12]} ({source_used})',
         f'Python {sys.version.split()[0]}, torch {VERSIONS["torch"]}, sklearn {VERSIONS["scikit-learn"]}, CPUs {os.cpu_count()}, execuções em paralelo {N_PARALLEL}',
         f'Atributos: {FEATURE_DIM} (INCLUDE_ID_FEATURES={INCLUDE_ID_FEATURES}) | sementes finais {SEEDS} | sementes de busca {TUNE_SEEDS + CONFIRM_EXTRA_SEEDS}',
         '', '## Dataset', md_table(pd.Series(DATASET_STATS, name='valor').to_frame()),
         '', '## Divisão (nós, fraudes)', md_table(pd.DataFrame(SPLIT, index=['nós', 'fraudes']).T),
         '', '## Grafo', md_table(GRAPH_STATS, index=False),
         '', '## Espaço de busca', json.dumps(SPACE),
         '', '## Busca (AUC de validação, 1 semente de busca por configuração)', md_table(search_table),
         '', f'## Confirmação (top {TOP_K}, sementes {TUNE_SEEDS + CONFIRM_EXTRA_SEEDS})', md_table(confirm_table),
         '', f'Configuração escolhida: {BEST_TAG} = {json.dumps(BEST_PARAMS)}',
         '', f'## Teste, {len(SEEDS)} sementes (média ± dp)', md_table(SUMMARY),
         '', '## Pareado ajustado − padrão', md_table(PAIRED, index=False),
         '', '## Por semente: padrão', md_table(df_base.round(4), index=False),
         '', '## Por semente: ajustado', md_table(df_tuned.round(4), index=False),
         '', '## Custo', md_table(cost),
         '', '=== FIM ===']
SUMMARY_TEXT = '\n'.join(lines)
(REPORT / 'resumo_para_enviar.md').write_text(SUMMARY_TEXT, encoding='utf-8')
print(SUMMARY_TEXT)

=== RESUMO S-FFSD (copie daqui até FIM) ===

Commit PC-GNN: 9d7d7fae491081178b6e12e193fb89cfc330f2be | adaptação +72/-52 linhas | S-FFSD sha256 a2d78b983dfa (download (AI4Risk/antifraud, ramo main))
Python 3.13.18, torch 2.14.1+cpu, sklearn 1.6.1, CPUs 2, execuções em paralelo 2
Atributos: 121 (INCLUDE_ID_FEATURES=False) | sementes finais [42, 448, 854, 29493, 88867, 72, 123, 2024, 31337, 7] | sementes de busca [1001, 1002, 1003]

## Dataset
| index | valor |
|---|---|
| nós (transações) | 77881.0 |
| nós rotulados | 29643.0 |
| nós sem rótulo | 48238.0 |
| fraudes (entre os rotulados) | 5256.0 |
| % de fraudes (rotulados) | 17.73 |
| IR entre os rotulados | 4.64 |
| Source distintos | 30346.0 |
| Target distintos | 886.0 |
| Location distintos | 296.0 |
| Type distintos | 166.0 |

## Divisão (nós, fraudes)
| index | nós | fraudes |
|---|---|---|
| treino | 11857 | 2102 |
| validação | 5869 | 1041 |
| teste | 11917 | 2113 |

## Grafo
| relação | arestas | grau médio | arestas entre rot